In [49]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [50]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.IRSwaps.IRSwapQuery import IRSwapQuery

In [ ]:
from SDRUtils.SDRDataBuilder import SDRDataBuilder
# from SDRUtils.classification import classify_trade, classifications_to_dataframe
from SDRUtils.usd_swaps.filters import new_sofr_swap_trades
# from SDRUtils.package_detection import (
#     detect_fly_trades_df,
#     detect_curve_trades_df,
#     detect_spreadover_trades_df,
#     _load_ust_reference_data,
#     merge_package_legs_to_one_row,
# )
from SDRUtils.seasonality import add_event_classifications, analyze_seasonality_by_event, aggregate_flows_by_label

In [52]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)

start = NY_tz.localize(datetime.datetime(2025, 12, 29, 00, 1))
end = NY_tz.localize(datetime.datetime(2025, 12, 29, 20, 00))

raw_df = sdr.grab_sdr_trades(
    start_timestamp=start, end_timestamp=end, agency="CFTC", asset_class="RATES", filter_func=new_sofr_swap_trades,
    # start_timestamp=start, end_timestamp=end, agency="CFTC", asset_class="RATES", filter_func=filter_new_sofr_swaption_trades,
    # start_timestamp=start, end_timestamp=end, agency="CFTC", asset_class="RATES",
)

swaps_mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-RL_BASIC")
curve = swaps_mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

CONCAT...: 100%|██████████| 2/2 [00:00<00:00, 309.45it/s]


In [53]:
raw_df

,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price currency,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name
0,1572688071000000201,NaN,NEWT,TRAD,2025-12-29 05:10:11+00:00,False,IR,None,I,True,...,,3.0,,,NaN,None,None,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
1,1572688070000000101,NaN,NEWT,TRAD,2025-12-29 05:10:11+00:00,False,IR,None,I,True,...,,3.0,,,NaN,None,None,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
2,1572688072000000301,NaN,NEWT,TRAD,2025-12-29 05:10:11+00:00,False,IR,None,I,True,...,,3.0,,,NaN,None,None,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
3,1572660519000000201,NaN,NEWT,TRAD,2025-12-29 05:10:11+00:00,False,IR,None,I,True,...,,3.0,,,NaN,None,None,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
4,1572660520000000301,NaN,NEWT,TRAD,2025-12-29 05:10:11+00:00,False,IR,None,I,True,...,,3.0,,,NaN,None,None,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2116,1574276291000000101,NaN,NEWT,TRAD,2025-12-30 00:11:10+00:00,NaN,IR,NaN,I,True,...,NaN,3.0,NaN,NaN,NaN,NaN,NaN,QZPB5VSBGRCD,NA/Swap OIS USD,USD-SOFR-OIS Compound
2117,1574395035000000101,NaN,NEWT,TRAD,2025-12-30 00:32:12+00:00,NaN,IR,NaN,I,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
2118,1574412460000000101,NaN,NEWT,TRAD,2025-12-30 00:40:19+00:00,NaN,IR,NaN,I,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
2119,1574434794000000101,NaN,NEWT,TRAD,2025-12-30 00:47:18+00:00,NaN,IR,NaN,I,True,...,NaN,NaN,-0.00365,NaN,3.0,NaN,NaN,QZPB5VSBGRCD,NA/Swap OIS USD,USD-SOFR-OIS Compound


In [5]:
from tqdm import tqdm

classifications = []
it = raw_df.iterrows()

it = tqdm(it, total=len(raw_df), desc="Classifying trades", unit="trade")

for idx, row in it:
	trade_id = int(row.get("Dissemination Identifier", idx))
	try:
		classification = classify_trade(row, trade_id, curve)
		classifications.append(classification)
	except Exception as e:
		continue

classifications_df = classifications_to_dataframe(classifications)
with_pkg_df = merge_package_legs_to_one_row(detect_spreadover_trades_df(detect_curve_trades_df(detect_fly_trades_df(classifications_df))))
# with_pkg_and_events_df = add_event_classifications(with_pkg_df, include_me=True)

Classifying trades: 100%|██████████| 1325/1325 [00:06<00:00, 205.44trade/s]


In [14]:
# with_pkg_df["estimated_pv01"].sort_values(key=lambda x: float(str(x).split("/")[0]) if type(x) == str else float(x))

with_pkg_df["pv01_clean"] = with_pkg_df["estimated_pv01"].apply(lambda x: float(str(x).split("/")[0]) if type(x) == str else float(x))
with_pkg_df.sort_values(by="pv01_clean", ascending=False).head(10)

,trade_id,execution_timestamp,effective_date,expiration_date,product_type,tenor_years,tenor_label,is_forward,forward_start_years,forward_label,...,estimated_pv01,package_type,package_id,package_legs,matched_ust_maturity,ust_cusip,ust_oi,ust_issue_date,swap_maturity_date,pv01_clean
379,1570969671000000201,2025-12-29 16:02:51+00:00,2025-12-31,2035-12-31 00:00:00,OIS_SWAP,10.144444,10Y,False,0.005556,spot,...,351955.397698,OUTRIGHT,None,None,False,NaN,NaN,NaN,2035-12-31,351955.397698
378,1570983813000000601,2025-12-29 16:01:00+00:00,2025-12-31,2045-12-31 00:00:00,OIS_SWAP,20.286111,20Y,False,0.005556,spot,...,276062.582362,OUTRIGHT,None,None,False,NaN,NaN,NaN,2045-12-31,276062.582362
362,1570963093000000101,2025-12-29 15:47:26+00:00,2025-12-31,2045-12-29 00:00:00,OIS_SWAP,20.286111,20Y,False,0.005556,spot,...,276062.582362,OUTRIGHT,None,None,False,NaN,NaN,NaN,2045-12-29,276062.582362
523,1572091922000000501,2025-12-29 17:51:29+00:00,2025-12-29,2055-11-17 00:00:00,OIS_SWAP,30.319444,30Y,False,0.000000,spot,...,258866.516903,OUTRIGHT,None,None,False,NaN,NaN,NaN,2055-11-17,258866.516903
542,1572445744000000301,2025-12-29 18:16:54+00:00,2025-12-29,2055-11-17 00:00:00,OIS_SWAP,30.319444,30Y,False,0.000000,spot,...,258866.516903,OUTRIGHT,None,None,False,NaN,NaN,NaN,2055-11-17,258866.516903
86,1570484968000001201,2025-12-29 12:09:34+00:00,2026-03-18,2056-03-18 00:00:00,OIS_SWAP,30.444444,IMM_H2056,True,0.219444,IMM_H2026,...,257064.268207,OUTRIGHT,None,None,False,NaN,NaN,NaN,2056-03-18,257064.268207
854,1569993470000000701,2025-12-29 11:34:00+00:00,2026-03-31,2032-11-30 00:00:00,OIS_SWAP,6.766667,7Y,True,0.255556,3M,...,253561.638471,SPREADOVER,SPREADOVER_1569993470000000701,[1569993470000000701],True,91282CPM7,7-Year,2025-12-01,2032-11-30,253561.638471
853,1569964814000000701,2025-12-29 11:34:00+00:00,2026-03-31,2032-11-30 00:00:00,OIS_SWAP,6.766667,7Y,True,0.255556,3M,...,253561.638471,SPREADOVER,SPREADOVER_1569964814000000701,[1569964814000000701],True,91282CPM7,7-Year,2025-12-01,2032-11-30,253561.638471
986,1570905700000000801,2025-12-29 15:13:22+00:00,2026-03-31,2035-08-15 00:00:00,OIS_SWAP,9.511111,10Y,True,0.255556,3M,...,252223.685872,SPREADOVER,SPREADOVER_1570905700000000801,[1570905700000000801],True,91282CNT4,10-Year,2025-08-15,2035-08-15,252223.685872
1053,1570993333000000101,2025-12-29 16:19:06+00:00,2026-03-31,2032-11-15 00:00:00,OIS_SWAP,6.725,7Y,True,0.255556,3M,...,252213.461277,SPREADOVER,SPREADOVER_1570993333000000101,[1570993333000000101],True,91282CFV8,10-Year,2022-11-15,2032-11-15,252213.461277


In [52]:
# with_pkg_df[with_pkg_df["effective_date"].dt.date == datetime.date(2026, 3, 31)]
with_pkg_df[with_pkg_df["package_type"] == "SPREADOVER"]["forward_label"].value_counts()

# with_pkg_df[(with_pkg_df["is_forward"] == True) & (with_pkg_df["forward_label"] == "IMM_H2026") & (with_pkg_df["package_type"] == "SPREADOVER0")].tail(20)

KeyError: 'package_type'

In [ ]:
swaps_mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-RL_BASIC")

curve = swaps_mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=datetime.date(2025, 12, 29)))

swap_query = IRSwapQuery(curve="USD-SOFR-1D", effective_date=datetime.date(2026, 3, 1), maturity_date=datetime.date(2043, 11, 15), structure_kwargs={"notional": 79_000_000})

pkg, rws = swap_query.resolve_package(pricer_or_curve=curve)

curve.pv01(pkg[0])

In [33]:
raw_df[raw_df["Dissemination Identifier"].isin([1571746453000000201	])].to_dict(orient="records")

[{'Dissemination Identifier': 1571746453000000201,
  'Original Dissemination Identifier': nan,
  'Action type': 'NEWT',
  'Event type': 'TRAD',
  'Event timestamp': Timestamp('2025-12-29 15:57:09+0000', tz='UTC'),
  'Amendment indicator': nan,
  'Asset Class': 'IR',
  'Product name': nan,
  'Cleared': 'I',
  'Mandatory clearing indicator': True,
  'Execution Timestamp': Timestamp('2025-12-29 15:57:09+0000', tz='UTC'),
  'Effective Date': Timestamp('2026-03-31 00:00:00'),
  'Expiration Date': Timestamp('2043-11-15 00:00:00'),
  'Maturity date of the underlier': '2043-11-15',
  'Non-standardized term indicator': nan,
  'Platform identifier': 'BILT',
  'Prime brokerage transaction indicator': False,
  'Block trade election indicator': False,
  'Large notional off-facility swap election indicator': False,
  'Notional amount-Leg 1': '79,000,000',
  'Notional amount-Leg 2': '79,000,000',
  'Notional currency-Leg 1': 'USD',
  'Notional currency-Leg 2': 'USD',
  'Notional quantity-Leg 1': nan,